# NIDQ Channel Configuration
Interactive widget for inspecting and labelling NIDQ channels from a SpikeGLX recording session.
Labels (name, type, units, notes) are saved to `session_config.json` and applied automatically on future `load_nidq()` calls.

## 1. Import Required Libraries

In [1]:
%matplotlib widget

from mimo_pack.core import Session
from mimo_pack.widgets import NidqChannelConfigWidget
from mimo_pack.fileio.nidq import load_nidq
from mimo_pack.consts import DIM_TIME, DIM_CHAN, ATTR_FS

## 2. Define Session Directory

In [2]:
from mimo_pack.util import create_session_config

# ---- Parameters ----
data_dir = r'D:\SortingTemp\24-05-31_DH23_A1-IC_g0'

# Create session_config.json if it doesn't already exist
create_session_config(data_dir)

sess = Session(data_dir)
print(f"Session dir   : {sess.dir}")
print(f"Config path   : {sess.config_path}")

# Find NIDQ bin files, then resolve to dclut JSON paths
nidq_bin_files = sess.find_files('nidq_pattern')
print(f"\nNIDQ bin files found ({len(nidq_bin_files)}):")
for f in nidq_bin_files:
    print(f"  {f}")

nidq_files = [sess.get_dclut_path(str(f)) for f in nidq_bin_files]
print(f"\nNIDQ dclut paths:")
for f in nidq_files:
    print(f"  {f}")


ERROR: Config file already exists at D:\SortingTemp\24-05-31_DH23_A1-IC_g0\session_config.json. Set overwrite=True to replace it.
Use --overwrite to replace the existing configuration.
Session dir   : D:\SortingTemp\24-05-31_DH23_A1-IC_g0
Config path   : D:\SortingTemp\24-05-31_DH23_A1-IC_g0\session_config.json

NIDQ bin files found (1):
  D:\SortingTemp\24-05-31_DH23_A1-IC_g0\24-05-31_DH23_A1-IC_g0_t0.nidq.bin

NIDQ dclut paths:
  D:\SortingTemp\24-05-31_DH23_A1-IC_g0\24-05-31_DH23_A1-IC_g0_t0.nidq_dclut.json


## 3. Load the NIDQ File
Load the raw NIDQ data to inspect its structure before launching the widget.

In [3]:
# Load the first NIDQ file (no channel renaming yet — widget will configure that)
nidq_path = str(nidq_files[0])
nidq = load_nidq(nidq_path)

fs      = float(nidq.attrs[ATTR_FS])
t_start = float(nidq[DIM_TIME].values[0])
t_end   = float(nidq[DIM_TIME].values[-1])
n_chans = nidq.sizes[DIM_CHAN]

print(f"File      : {nidq_path}")
print(f"Shape     : {nidq.shape}  dims={nidq.dims}")
print(f"Channels  : {n_chans}")
print(f"Fs        : {fs:.1f} Hz")
print(f"Duration  : {t_end - t_start:.1f} s  ({t_start:.3f} → {t_end:.3f} s)")
print(f"\nAttrs:")
for k, v in nidq.attrs.items():
    print(f"  {k}: {v}")

File      : D:\SortingTemp\24-05-31_DH23_A1-IC_g0\24-05-31_DH23_A1-IC_g0_t0.nidq_dclut.json
Shape     : (286142356, 24)  dims=('time', 'channel')
Channels  : 24
Fs        : 10593.3 Hz
Duration  : 27011.7 s  (0.000 → 27011.665 s)

Attrs:
  fs: 10593.28296979215
  description: NIDQ data: D:\SortingTemp\24-05-31_DH23_A1-IC_g0\24-05-31_DH23_A1-IC_g0_t0.nidq_dclut.json -> XD bits expanded


In [4]:
print(nidq)

<xarray.DataArray (time: 286142356, channel: 24)> Size: 27GB
array([[18656.,    51.,    53., ...,     0.,     0.,     0.],
       [18664.,    51.,    48., ...,     0.,     0.,     0.],
       [18665.,    48.,    51., ...,     0.,     0.,     0.],
       ...,
       [18638.,    62.,    67., ...,     0.,     0.,     0.],
       [18640.,    62.,    63., ...,     0.,     0.,     0.],
       [18644.,    61.,    63., ...,     0.,     0.,     0.]],
      shape=(286142356, 24), dtype=float32)
Coordinates:
  * time       (time) float64 2GB 0.0 9.436e-05 ... 2.701e+04 2.701e+04
  * channel    (channel) <U7 672B '0' '1' '2' ... 'XD0_b13' 'XD0_b14' 'XD0_b15'
    chan_type  (channel) <U7 672B 'analog' 'analog' ... 'digital' 'digital'
    ch_name    (channel) <U7 672B 'XA0;0' 'XA1;1' ... 'XD0_b14' 'XD0_b15'
Attributes:
    fs:           10593.28296979215
    description:  NIDQ data: D:\SortingTemp\24-05-31_DH23_A1-IC_g0\24-05-31_D...


## 4. Inspect Channel Data
Preview raw channel values and check data quality before configuring labels.

In [5]:
import pandas as pd

# Summary table: channel index, min, max, std of first 5 s
t_inspect = nidq.sel({DIM_TIME: slice(t_start, t_start + 5.0)})
rows = []
for ch in t_inspect[DIM_CHAN].values:
    ch_data = t_inspect.sel({DIM_CHAN: ch}).values
    rows.append({
        'channel': str(ch),
        'min':     float(ch_data.min()),
        'max':     float(ch_data.max()),
        'std':     float(ch_data.std()),
        'mean':    float(ch_data.mean()),
    })
summary_df = pd.DataFrame(rows).set_index('channel')
print("Channel summary (first 5 s):")
summary_df

Channel summary (first 5 s):


,min,max,std,mean
channel,,,,
0,18647.0,18673.0,3.712805,18662.136719
1,-17.0,92.0,7.070655,43.427795
2,-12.0,85.0,6.941234,43.566071
3,121.0,235.0,4.249221,170.269302
4,758.0,31428.0,15064.734375,13600.791016
5,894.0,31428.0,15009.596680,13704.998047
6,895.0,31424.0,14999.984375,13717.067383
7,888.0,31353.0,14959.832031,13692.384766
XD0_b00,0.0,0.0,0.000000,0.000000


## 5. Initialize and Interact with the Widget

The widget will:
1. **Auto-detect** the NIDQ dclut path from the session
2. **Overview** — plot all channels for the full session (downsampled)
3. **Channel Config table** — edit `name`, `type`, `units`, and `notes` per channel
4. **Preview** — scroll through any time window with an interactive slider
5. **Save to Config** — persist labels to `session_config.json`

After saving, future calls to `load_nidq(path, chan_names=...)` will apply the configured names.

In [6]:
nidq_widget = NidqChannelConfigWidget(sess)
nidq_widget.display()

## 6. Reload with Configured Channel Names
After saving in the widget, reload the NIDQ data with the configured channel names applied.

In [ ]:
import importlib, pandas as pd
import mimo_pack.fileio.nidq as _nidq_mod
import mimo_pack.fileio.spikeglx as _sglx_mod
importlib.reload(_nidq_mod)
importlib.reload(_sglx_mod)
from mimo_pack.fileio.nidq import load_nidq
from mimo_pack.fileio.spikeglx import dclut_from_meta

# ---- Re-generate dclut with per-channel gain scale ----
bin_path = str(nidq_bin_files[0])
print(f"Regenerating dclut for:\n  {bin_path}")
new_dclut_path = dclut_from_meta(bin_path)
print(f"Written: {new_dclut_path}\n")

# ---- Reload — gain applied automatically, units set to 'V' ----
nidq_v = load_nidq(new_dclut_path)

print(f"Shape  : {nidq_v.shape}  dims={nidq_v.dims}")
print(f"Units  : {nidq_v.attrs.get('units', '(not set)')}")
print(f"\nPer-channel stats (first 5 s):")
t0 = float(nidq_v[DIM_TIME].values[0])
t_win = nidq_v.sel({DIM_TIME: slice(t0, t0 + 5.0)})
rows_v = []
for ch in t_win[DIM_CHAN].values:
    ch_data = t_win.sel({DIM_CHAN: ch}).values
    rows_v.append({
        'channel':  str(ch),
        'chan_type': str(nidq_v.coords['chan_type'].sel({DIM_CHAN: ch}).values)
                    if 'chan_type' in nidq_v.coords else '',
        'min (V)':  float(ch_data.min()),
        'max (V)':  float(ch_data.max()),
        'std (V)':  float(ch_data.std()),
    })
pd.DataFrame(rows_v).set_index('channel')


Regenerating dclut for: D:\SortingTemp\24-05-31_DH23_A1-IC_g0\24-05-31_DH23_A1-IC_g0_t0.nidq.bin
Written: D:\SortingTemp\24-05-31_DH23_A1-IC_g0\24-05-31_DH23_A1-IC_g0_t0.nidq_dclut.json

Shape  : (286160384, 24)  dims=('time', 'channel')
Units  : (not set)

Per-channel stats (first 5 s):


,chan_type,min (V),max (V),std (V)
channel,,,,
0,analog,18647.0,18673.0,3.712891
1,analog,-17.0,92.0,7.070889
2,analog,-12.0,85.0,6.941525
3,analog,121.0,235.0,4.249225
4,analog,758.0,31428.0,15065.009766
5,analog,894.0,31428.0,15009.868164
6,analog,895.0,31424.0,15000.252930
7,analog,888.0,31353.0,14960.098633
XD0_b00,digital,0.0,0.0,0.000000


In [ ]:
# Re-read saved channel config from session (reload after widget saves)
sess_fresh = Session(data_dir)
saved_channels = sess_fresh.config.get('parameters', {}).get('nidq_channels', {})

if saved_channels:
    # Build chan_names dict: {original_id: label}  (skip channels with no name set)
    chan_names = {
        cid: cfg['name']
        for cid, cfg in saved_channels.items()
        if cfg.get('name', '')
    }
    print(f"Applying {len(chan_names)} channel labels:")
    for cid, name in chan_names.items():
        print(f"  channel {cid} → '{name}'")

    nidq_named = load_nidq(nidq_path, chan_names=chan_names)
    print(f"\nNamed DataArray channel coordinate:")
    print(nidq_named[DIM_CHAN].values)
else:
    print("No channel config saved yet — run the widget and click 'Save to Config' first.")
    nidq_named = nidq